                                        CELL 1

In [ ]:
# =========================================================
# EfficientNetV2 + ViT + Attention Fusion (TRAINED) + XGBoost
# Cross Validation + ROC-AUC + MCC + Ablation Ready
# Optimized for Apple Silicon (M1/M2)
#
# FIXES APPLIED vs previous version:
#   1. AttentionFusion is NOW TRAINED per fold (supervised,
#      using a small classification head). No more random weights.
#   2. Fusion layer re-initialized each fold — zero data leakage.
#   3. Best fold model (highest val AUC) is saved, not last fold.
# =========================================================
# =========================================================
# WITH CONFUSION MATRIX VISUALIZATION
# EfficientNetV2 + ViT + Attention Fusion + XGBoost
# =========================================================

import os, time, cv2, timm, torch, random, joblib, warnings
import numpy as np, pandas as pd, torch.nn as nn, torch.optim as optim

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns

import torchvision.transforms as transforms

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

from tqdm import tqdm
from PIL import Image
from pathlib import Path

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
    classification_report
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================

SEED              = 42
DATASET_PATH      = "/Users/shuvasishsaha/Documents/SAAMPLE"

NUM_CLASSES       = 2
NUM_FOLDS         = 5

BATCH_SIZE        = 4
FUSION_BATCH_SIZE = 4

FUSION_EPOCHS     = 10
FUSION_LR         = 1e-3

XGB_ESTIMATORS    = 300

# =========================================================
# REPRODUCIBILITY
# =========================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

print(f"Device: {device}")

# =========================================================
# TRANSFORMS
# =========================================================

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

# =========================================================
# OUTPUT DIRECTORIES
# =========================================================

os.makedirs("outputs/models", exist_ok=True)
os.makedirs("outputs/confusion_matrices", exist_ok=True)

print("Config ready.")

# =========================================================
# LOAD DATASET
# =========================================================

full_dataset = ImageFolder(DATASET_PATH)

class_names = full_dataset.classes
targets     = full_dataset.targets

print("\nClasses:", class_names)
print("Total Images:", len(full_dataset))

# =========================================================
# ATTENTION FUSION MODULE
# =========================================================

class AttentionFusion(nn.Module):

    def __init__(self, cnn_dim, vit_dim, num_classes=2):

        super().__init__()

        self.total_dim = cnn_dim + vit_dim

        self.attention = nn.Sequential(
            nn.Linear(self.total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, self.total_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.total_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, cnn_feat, vit_feat, return_logits=False):

        combined = torch.cat([cnn_feat, vit_feat], dim=1)

        weights = self.attention(combined)

        fused = combined * weights

        if return_logits:
            return self.classifier(fused)

        return fused

# =========================================================
# LOAD BACKBONE MODELS
# =========================================================

print("\nLoading pretrained backbone models...")

cnn_model = timm.create_model(
    'tf_efficientnetv2_s',
    pretrained=True,
    num_classes=0
)

vit_model = timm.create_model(
    'vit_small_patch16_224',
    pretrained=True,
    num_classes=0
)

for p in cnn_model.parameters():
    p.requires_grad = False

for p in vit_model.parameters():
    p.requires_grad = False

cnn_model.to(device).eval()
vit_model.to(device).eval()

# =========================================================
# FEATURE DIMENSIONS
# =========================================================

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():

    cnn_dim = cnn_model(dummy).shape[1]
    vit_dim = vit_model(dummy).shape[1]

print(f"\nCNN Feature Dim : {cnn_dim}")
print(f"ViT Feature Dim : {vit_dim}")
print(f"Fused Dim       : {cnn_dim + vit_dim}")

# =========================================================
# PARAMETER COUNTS
# =========================================================

cnn_params = sum(p.numel() for p in cnn_model.parameters()) / 1e6
vit_params = sum(p.numel() for p in vit_model.parameters()) / 1e6

print(f"\nEfficientNetV2-S : {cnn_params:.1f}M parameters")
print(f"ViT-S/16         : {vit_params:.1f}M parameters")

# =========================================================
# FUSION TRAINING
# =========================================================

def train_fusion_layer(fusion_layer, train_loader, val_loader):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        fusion_layer.parameters(),
        lr=FUSION_LR,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=FUSION_EPOCHS
    )

    best_val_loss = float('inf')
    best_state = None

    for epoch in range(FUSION_EPOCHS):

        # ---------------- TRAIN ----------------

        fusion_layer.train()

        train_loss = 0.0

        train_pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{FUSION_EPOCHS} [Train]",
            leave=False
        )

        for images, labels in train_pbar:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.no_grad():

                cnn_feat = cnn_model(images)
                vit_feat = vit_model(images)

            logits = fusion_layer(
                cnn_feat,
                vit_feat,
                return_logits=True
            )

            loss = criterion(logits, labels)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

            train_pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        scheduler.step()

        # ---------------- VALIDATION ----------------

        fusion_layer.eval()

        val_loss = 0.0

        val_pbar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{FUSION_EPOCHS} [Val]",
            leave=False
        )

        with torch.no_grad():

            for images, labels in val_pbar:

                images = images.to(device)
                labels = labels.to(device)

                cnn_feat = cnn_model(images)
                vit_feat = vit_model(images)

                logits = fusion_layer(
                    cnn_feat,
                    vit_feat,
                    return_logits=True
                )

                loss = criterion(logits, labels)

                val_loss += loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss / len(val_loader)

        print(
            f"Epoch [{epoch+1:02d}/{FUSION_EPOCHS}] "
            f"Train Loss: {avg_train:.4f}  "
            f"Val Loss: {avg_val:.4f}"
        )

        if avg_val < best_val_loss:

            best_val_loss = avg_val

            best_state = {
                k: v.clone()
                for k, v in fusion_layer.state_dict().items()
            }

    if best_state is not None:
        fusion_layer.load_state_dict(best_state)

    fusion_layer.eval()

    print(f"Best validation loss: {best_val_loss:.4f}")

    return fusion_layer

# =========================================================
# FEATURE EXTRACTION
# =========================================================

@torch.no_grad()
def extract_features(loader, fusion_layer):

    all_features = []
    all_labels   = []

    fusion_layer.eval()

    for images, labels in tqdm(loader, leave=False):

        images = images.to(device)

        cnn_feat = cnn_model(images)
        vit_feat = vit_model(images)

        fused = fusion_layer(
            cnn_feat,
            vit_feat,
            return_logits=False
        )

        all_features.append(fused.cpu().numpy())
        all_labels.append(labels.numpy())

    return (
        np.concatenate(all_features),
        np.concatenate(all_labels)
    )

# =========================================================
# CROSS VALIDATION
# =========================================================

kf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# METRIC STORAGE
# =========================================================

all_accuracy    = []
all_precision   = []
all_recall      = []
all_f1          = []
all_auc         = []
all_mcc         = []
all_kappa       = []
all_specificity = []
all_fold_times  = []

all_y_true = []
all_y_pred = []

best_auc       = -1.0
best_xgb_model = None
best_fold_idx  = -1

# =========================================================
# MAIN TRAINING LOOP
# =========================================================

pipeline_start = time.time()

for fold, (train_idx, val_idx) in enumerate(
    kf.split(np.arange(len(targets)), targets)
):

    fold_start = time.time()

    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/{NUM_FOLDS}")
    print(f"{'='*50}")

    # =====================================================
    # FRESH FUSION LAYER
    # =====================================================

    fusion_layer = AttentionFusion(
        cnn_dim,
        vit_dim,
        NUM_CLASSES
    ).to(device)

    # =====================================================
    # DATA LOADERS
    # =====================================================

    train_subset = Subset(
        ImageFolder(DATASET_PATH, transform=train_transform),
        train_idx
    )

    val_subset = Subset(
        ImageFolder(DATASET_PATH, transform=val_transform),
        val_idx
    )

    train_loader = DataLoader(
        train_subset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    # =====================================================
    # TRAIN FUSION
    # =====================================================

    print("\n[Step 1] Training fusion layer...")

    fusion_layer = train_fusion_layer(
        fusion_layer,
        train_loader,
        val_loader
    )

    # =====================================================
    # FEATURE EXTRACTION
    # =====================================================

    print("\n[Step 2] Extracting features...")

    X_train, y_train = extract_features(
        train_loader,
        fusion_layer
    )

    X_val, y_val = extract_features(
        val_loader,
        fusion_layer
    )

    print(f"Train shape: {X_train.shape}")
    print(f"Val shape  : {X_val.shape}")

    # =====================================================
    # XGBOOST
    # =====================================================

    print("\n[Step 3] Training XGBoost...")

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=SEED,
        tree_method='hist'
    )

    xgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # =====================================================
    # EVALUATION
    # =====================================================

    y_pred = xgb_model.predict(X_val)
    y_prob = xgb_model.predict_proba(X_val)[:,1]

    all_y_true.extend(y_val)
    all_y_pred.extend(y_pred)

    accuracy  = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall    = recall_score(y_val, y_pred, zero_division=0)
    f1        = f1_score(y_val, y_pred, zero_division=0)
    auc       = roc_auc_score(y_val, y_prob)

    mcc   = matthews_corrcoef(y_val, y_pred)
    kappa = cohen_kappa_score(y_val, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    specificity = tn / (tn + fp)

    all_accuracy.append(accuracy)
    all_precision.append(precision)
    all_recall.append(recall)
    all_f1.append(f1)
    all_auc.append(auc)
    all_mcc.append(mcc)
    all_kappa.append(kappa)
    all_specificity.append(specificity)

    fold_time = time.time() - fold_start

    all_fold_times.append(fold_time)

    # =====================================================
    # PRINT RESULTS
    # =====================================================

    print(f"\n===== Fold {fold+1} Results =====")

    print(f"Accuracy   : {accuracy:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print(f"ROC-AUC    : {auc:.4f}")
    print(f"MCC        : {mcc:.4f}")
    print(f"Kappa      : {kappa:.4f}")
    print(f"Specificity: {specificity:.4f}")

    print("\nClassification Report:\n")

    print(
        classification_report(
            y_val,
            y_pred,
            target_names=class_names
        )
    )

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    cm = confusion_matrix(y_val, y_pred)

    plt.figure(figsize=(6,5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    plt.title(f"Confusion Matrix - Fold {fold+1}")

    plt.savefig(
        f"outputs/confusion_matrices/fold_{fold+1}_cm.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    print(f"Confusion matrix saved for Fold {fold+1}")

    # =====================================================
    # SAVE BEST MODEL
    # =====================================================

    if auc > best_auc:

        best_auc       = auc
        best_xgb_model = xgb_model
        best_fold_idx  = fold + 1

        torch.save(
            fusion_layer.state_dict(),
            "outputs/models/best_fusion_layer.pt"
        )

        print(
            f"\n>>> New best model saved "
            f"(Fold {fold+1}, AUC={auc:.4f})"
        )

# =========================================================
# FINAL RESULTS
# =========================================================

total_time = time.time() - pipeline_start

print("\n" + "="*50)
print("FINAL CROSS-VALIDATION RESULTS")
print("="*50)

print(f"\nAccuracy    : {np.mean(all_accuracy):.4f} ± {np.std(all_accuracy):.4f}")
print(f"Precision   : {np.mean(all_precision):.4f} ± {np.std(all_precision):.4f}")
print(f"Recall      : {np.mean(all_recall):.4f} ± {np.std(all_recall):.4f}")
print(f"F1 Score    : {np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}")
print(f"ROC-AUC     : {np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}")
print(f"MCC         : {np.mean(all_mcc):.4f} ± {np.std(all_mcc):.4f}")
print(f"Kappa       : {np.mean(all_kappa):.4f} ± {np.std(all_kappa):.4f}")
print(f"Specificity : {np.mean(all_specificity):.4f} ± {np.std(all_specificity):.4f}")

print(f"\nBest Fold : {best_fold_idx}")
print(f"Best AUC  : {best_auc:.4f}")

# =========================================================
# FINAL AGGREGATED CONFUSION MATRIX
# =========================================================

final_cm = confusion_matrix(all_y_true, all_y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    final_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.title("Final Aggregated Confusion Matrix")

plt.savefig(
    "outputs/final_confusion_matrix.png",
    dpi=300,
    bbox_inches='tight'
)

plt.close()

print("\nFinal confusion matrix saved.")

# =========================================================
# SAVE BEST XGB MODEL
# =========================================================

joblib.dump(
    best_xgb_model,
    "outputs/models/best_xgb_model.pkl"
)

print("\nBest XGBoost model saved.")

# =========================================================
# SAVE RESULTS CSV
# =========================================================

results = pd.DataFrame({
    "Fold"       : list(range(1, NUM_FOLDS + 1)),
    "Accuracy"   : all_accuracy,
    "Precision"  : all_precision,
    "Recall"     : all_recall,
    "F1"         : all_f1,
    "ROC_AUC"    : all_auc,
    "MCC"        : all_mcc,
    "Kappa"      : all_kappa,
    "Specificity": all_specificity,
    "Time_min"   : [t/60 for t in all_fold_times]
})

results.to_csv(
    "outputs/fold_results.csv",
    index=False
)

print("Results CSV saved.")

# =========================================================
# COMPUTATIONAL COST REPORT
# =========================================================

print("\n" + "="*50)
print("COMPUTATIONAL COST")
print("="*50)

print(f"\nHardware          : Apple Silicon (MPS)")
print(f"EfficientNetV2-S  : {cnn_params:.1f}M parameters")
print(f"ViT-S/16          : {vit_params:.1f}M parameters")
print(f"Fusion epochs     : {FUSION_EPOCHS}")
print(f"Total pipeline    : {total_time/60:.1f} min")
print(f"Average/fold      : {np.mean(all_fold_times)/60:.1f} min")

print("\nPipeline completed successfully.")

Device: mps
Config ready.

Classes: ['healthy', 'infected']
Total Images: 204

Loading pretrained backbone models...


KeyboardInterrupt: 

                                                CELL 2

In [ ]:
# =========================================================
# WITH FULL TQDM TIMERS + PROGRESS TRACKING
#
# Config 1: EfficientNetV2-S only + XGBoost
# Config 2: ViT-S/16 only + XGBoost
# Config 3: EfficientNet + ViT (concat) + XGBoost
# Config 4: EfficientNet + ViT + Attention Fusion + XGBoost
#
# Outputs:
# - Full ablation CSV
# - Fold-wise timing
# - tqdm progress bars
# - Runtime analysis
# =========================================================

import os
import time
import timm
import torch
import random
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix
)

from xgboost import XGBClassifier

import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================

DATASET_PATH  = "/Users/shuvasishsaha/Documents/SAAMPLE"

SEED          = 42
NUM_FOLDS     = 5

BATCH_SIZE    = 4

FUSION_EPOCHS = 10
FUSION_LR     = 1e-3

NUM_CLASSES   = 2
XGB_ESTIMATORS = 300

# =========================================================
# REPRODUCIBILITY
# =========================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

print(f"\nDevice: {device}")

# =========================================================
# TRANSFORMS
# =========================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# =========================================================
# LOAD DATASET
# =========================================================

full_dataset = ImageFolder(DATASET_PATH)

targets     = full_dataset.targets
class_names = full_dataset.classes

print(f"\nClasses      : {class_names}")
print(f"Total Images : {len(full_dataset)}")

# =========================================================
# LOAD BACKBONE MODELS
# =========================================================

print("\nLoading pretrained backbone models...")

cnn_model = timm.create_model(
    'tf_efficientnetv2_s',
    pretrained=True,
    num_classes=0
)

vit_model = timm.create_model(
    'vit_small_patch16_224',
    pretrained=True,
    num_classes=0
)

# Freeze backbone parameters
for p in cnn_model.parameters():
    p.requires_grad = False

for p in vit_model.parameters():
    p.requires_grad = False

cnn_model.to(device).eval()
vit_model.to(device).eval()

# =========================================================
# FEATURE DIMENSIONS
# =========================================================

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    cnn_dim = cnn_model(dummy).shape[1]
    vit_dim = vit_model(dummy).shape[1]

print(f"\nCNN Feature Dim : {cnn_dim}")
print(f"ViT Feature Dim : {vit_dim}")
print(f"Concat Dim      : {cnn_dim + vit_dim}")

# =========================================================
# PARAMETER COUNTS
# =========================================================

cnn_params = sum(p.numel() for p in cnn_model.parameters()) / 1e6
vit_params = sum(p.numel() for p in vit_model.parameters()) / 1e6

print(f"\nEfficientNetV2-S : {cnn_params:.1f}M parameters")
print(f"ViT-S/16         : {vit_params:.1f}M parameters")

# =========================================================
# ATTENTION FUSION MODULE
# =========================================================

class AttentionFusion(nn.Module):

    def __init__(self, cnn_dim, vit_dim, num_classes=2):

        super().__init__()

        self.total_dim = cnn_dim + vit_dim

        self.attention = nn.Sequential(
            nn.Linear(self.total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, self.total_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.total_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, cnn_feat, vit_feat, return_logits=False):

        combined = torch.cat([cnn_feat, vit_feat], dim=1)

        weights = self.attention(combined)

        fused = combined * weights

        if return_logits:
            return self.classifier(fused)

        return fused

# =========================================================
# TRAIN FUSION LAYER
# =========================================================

def train_fusion(fusion_layer, train_loader, val_loader):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        fusion_layer.parameters(),
        lr=FUSION_LR,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=FUSION_EPOCHS
    )

    best_val_loss = float('inf')
    best_state = None

    for epoch in range(FUSION_EPOCHS):

        # -------------------------------------------------
        # TRAIN
        # -------------------------------------------------

        fusion_layer.train()

        train_loss = 0.0

        train_pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{FUSION_EPOCHS} [Train]",
            leave=False
        )

        for images, labels in train_pbar:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                cnn_feat = cnn_model(images)
                vit_feat = vit_model(images)

            logits = fusion_layer(
                cnn_feat,
                vit_feat,
                return_logits=True
            )

            loss = criterion(logits, labels)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

            train_pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        scheduler.step()

        # -------------------------------------------------
        # VALIDATION
        # -------------------------------------------------

        fusion_layer.eval()

        val_loss = 0.0

        val_pbar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{FUSION_EPOCHS} [Val]",
            leave=False
        )

        with torch.no_grad():

            for images, labels in val_pbar:

                images = images.to(device)
                labels = labels.to(device)

                cnn_feat = cnn_model(images)
                vit_feat = vit_model(images)

                logits = fusion_layer(
                    cnn_feat,
                    vit_feat,
                    return_logits=True
                )

                loss = criterion(logits, labels)

                val_loss += loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss / len(val_loader)

        print(
            f"  Epoch [{epoch+1:02d}/{FUSION_EPOCHS}] "
            f"Train Loss: {avg_train:.4f}  "
            f"Val Loss: {avg_val:.4f}"
        )

        if avg_val < best_val_loss:

            best_val_loss = avg_val

            best_state = {
                k: v.clone()
                for k, v in fusion_layer.state_dict().items()
            }

    if best_state is not None:
        fusion_layer.load_state_dict(best_state)

    fusion_layer.eval()

    print(f"  ✅ Best Val Loss: {best_val_loss:.4f}")

    return fusion_layer

# =========================================================
# FEATURE EXTRACTORS
# =========================================================

@torch.no_grad()
def extract_cnn_only(loader):

    X, y = [], []

    for imgs, labels in tqdm(
        loader,
        desc="Extract CNN Features",
        leave=False
    ):

        feats = cnn_model(imgs.to(device))

        X.append(feats.cpu().numpy())
        y.append(labels.numpy())

    return np.concatenate(X), np.concatenate(y)

@torch.no_grad()
def extract_vit_only(loader):

    X, y = [], []

    for imgs, labels in tqdm(
        loader,
        desc="Extract ViT Features",
        leave=False
    ):

        feats = vit_model(imgs.to(device))

        X.append(feats.cpu().numpy())
        y.append(labels.numpy())

    return np.concatenate(X), np.concatenate(y)

@torch.no_grad()
def extract_concat(loader):

    X, y = [], []

    for imgs, labels in tqdm(
        loader,
        desc="Extract Concat Features",
        leave=False
    ):

        imgs = imgs.to(device)

        feat = torch.cat([
            cnn_model(imgs),
            vit_model(imgs)
        ], dim=1)

        X.append(feat.cpu().numpy())
        y.append(labels.numpy())

    return np.concatenate(X), np.concatenate(y)

@torch.no_grad()
def extract_fusion(loader, fusion_layer):

    X, y = [], []

    fusion_layer.eval()

    for imgs, labels in tqdm(
        loader,
        desc="Extract Fusion Features",
        leave=False
    ):

        imgs = imgs.to(device)

        fused = fusion_layer(
            cnn_model(imgs),
            vit_model(imgs),
            return_logits=False
        )

        X.append(fused.cpu().numpy())
        y.append(labels.numpy())

    return np.concatenate(X), np.concatenate(y)

# =========================================================
# METRICS
# =========================================================

def compute_metrics(y_true, y_pred, y_prob):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        ),

        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        ),

        "Kappa": cohen_kappa_score(
            y_true,
            y_pred
        ),

        "Specificity": tn / (tn + fp)
    }

# =========================================================
# ABLATION CONFIGS
# =========================================================

configs = [

    {
        "name": "EfficientNetV2-S only",
        "extractor": "cnn"
    },

    {
        "name": "ViT-S/16 only",
        "extractor": "vit"
    },

    {
        "name": "EfficientNet + ViT (concat)",
        "extractor": "concat"
    },

    {
        "name": "EfficientNet + ViT + Attention (proposed)",
        "extractor": "fusion"
    }
]

# =========================================================
# CROSS VALIDATION
# =========================================================

kf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=SEED
)

ablation_rows = []

# =========================================================
# MAIN LOOP
# =========================================================

pipeline_start = time.time()

for cfg in configs:

    config_start = time.time()

    print(f"\n{'='*60}")
    print(f"CONFIGURATION: {cfg['name']}")
    print(f"{'='*60}")

    fold_metrics = []
    fold_times   = []

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(np.arange(len(targets)), targets)
    ):

        fold_start = time.time()

        print(f"\nFold {fold+1}/{NUM_FOLDS}")

        train_sub = Subset(
            ImageFolder(DATASET_PATH, transform=train_transform),
            train_idx
        )

        val_sub = Subset(
            ImageFolder(DATASET_PATH, transform=val_transform),
            val_idx
        )

        train_loader = DataLoader(
            train_sub,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0
        )

        val_loader = DataLoader(
            val_sub,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        ext = cfg["extractor"]

        # =================================================
        # FEATURE EXTRACTION
        # =================================================

        print("\n[Step 1] Feature Extraction")

        if ext == "cnn":

            X_train, y_train = extract_cnn_only(train_loader)
            X_val, y_val     = extract_cnn_only(val_loader)

        elif ext == "vit":

            X_train, y_train = extract_vit_only(train_loader)
            X_val, y_val     = extract_vit_only(val_loader)

        elif ext == "concat":

            X_train, y_train = extract_concat(train_loader)
            X_val, y_val     = extract_concat(val_loader)

        elif ext == "fusion":

            print("\n[Step 1A] Training Fusion Layer")

            fusion_layer = AttentionFusion(
                cnn_dim,
                vit_dim,
                NUM_CLASSES
            ).to(device)

            fusion_layer = train_fusion(
                fusion_layer,
                train_loader,
                val_loader
            )

            print("\n[Step 1B] Extracting Fusion Features")

            X_train, y_train = extract_fusion(
                train_loader,
                fusion_layer
            )

            X_val, y_val = extract_fusion(
                val_loader,
                fusion_layer
            )

        print(f"Train Shape: {X_train.shape}")
        print(f"Val Shape  : {X_val.shape}")

        # =================================================
        # XGBOOST
        # =================================================

        print("\n[Step 2] Training XGBoost")

        xgb = XGBClassifier(
            n_estimators=XGB_ESTIMATORS,
            max_depth=6,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=SEED,
            tree_method='hist'
        )

        xgb.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        # =================================================
        # EVALUATION
        # =================================================

        print("\n[Step 3] Evaluation")

        y_pred = xgb.predict(X_val)
        y_prob = xgb.predict_proba(X_val)[:, 1]

        metrics = compute_metrics(
            y_val,
            y_pred,
            y_prob
        )

        fold_metrics.append(metrics)

        fold_time = time.time() - fold_start
        fold_times.append(fold_time)

        print(f"\nAUC       : {metrics['ROC_AUC']:.4f}")
        print(f"F1 Score  : {metrics['F1']:.4f}")
        print(f"Accuracy  : {metrics['Accuracy']:.4f}")
        print(f"Fold Time : {fold_time/60:.1f} min")

    # =====================================================
    # CONFIG AVERAGES
    # =====================================================

    avg = {
        k: np.mean([fm[k] for fm in fold_metrics])
        for k in fold_metrics[0]
    }

    std = {
        k: np.std([fm[k] for fm in fold_metrics])
        for k in fold_metrics[0]
    }

    row = {
        "Configuration": cfg["name"]
    }

    for k in avg:

        row[k] = f"{avg[k]:.4f}"

        row[f"{k}_std"] = f"{std[k]:.4f}"

    row["Avg_Time_Min"] = f"{np.mean(fold_times)/60:.2f}"

    ablation_rows.append(row)

    config_time = time.time() - config_start

    print(f"\n{'-'*60}")
    print(f"CONFIG COMPLETE: {cfg['name']}")
    print(f"{'-'*60}")

    print(f"Mean AUC  : {avg['ROC_AUC']:.4f} ± {std['ROC_AUC']:.4f}")
    print(f"Mean F1   : {avg['F1']:.4f} ± {std['F1']:.4f}")
    print(f"Avg Time  : {np.mean(fold_times)/60:.2f} min/fold")
    print(f"Total CFG : {config_time/60:.1f} min")

# =========================================================
# SAVE RESULTS
# =========================================================

os.makedirs("outputs", exist_ok=True)

df = pd.DataFrame(ablation_rows)

df.to_csv(
    "outputs/ablation_results.csv",
    index=False
)

# =========================================================
# FINAL TABLE
# =========================================================

print("\n" + "="*90)
print("ABLATION TABLE (COPY INTO PAPER)")
print("="*90)

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "MCC",
    "Kappa",
    "Specificity"
]

header = f"{'Configuration':<45}" + "".join(
    f"{m:>12}" for m in metrics
)

print(header)
print("-" * len(header))

for row in ablation_rows:

    name = row["Configuration"]

    vals = "".join(
        f"{row[m]:>12}" for m in metrics
    )

    print(f"{name:<45}{vals}")

# =========================================================
# COMPUTATIONAL REPORT
# =========================================================

total_time = time.time() - pipeline_start

print("\n" + "="*60)
print("COMPUTATIONAL COST REPORT")
print("="*60)

print(f"\nHardware             : Apple M1 (MPS backend), 8GB RAM")
print(f"EfficientNetV2-S     : {cnn_params:.1f}M params")
print(f"ViT-S/16             : {vit_params:.1f}M params")
print(f"Fusion Training      : {FUSION_EPOCHS} epochs")
print(f"Cross Validation     : {NUM_FOLDS}-fold")
print(f"Total Runtime        : {total_time/60:.1f} min")

print("\nAblation results saved → outputs/ablation_results.csv")
print("\nAblation study completed successfully.")


Device: mps

Classes      : ['healthy', 'infected']
Total Images : 204

Loading pretrained backbone models...

CNN Feature Dim : 1280
ViT Feature Dim : 384
Concat Dim      : 1664

EfficientNetV2-S : 20.2M parameters
ViT-S/16         : 21.7M parameters

CONFIGURATION: EfficientNetV2-S only

Fold 1/5

[Step 1] Feature Extraction


Train Shape: (163, 1280)
Val Shape  : (41, 1280)

[Step 2] Training XGBoost

[Step 3] Evaluation

AUC       : 0.8643
F1 Score  : 0.7179
Accuracy  : 0.7317
Fold Time : 0.4 min

Fold 2/5

[Step 1] Feature Extraction


Train Shape: (163, 1280)
Val Shape  : (41, 1280)

[Step 2] Training XGBoost

[Step 3] Evaluation

AUC       : 0.7262
F1 Score  : 0.5946
Accuracy  : 0.6341
Fold Time : 0.4 min

Fold 3/5

[Step 1] Feature Extraction


Train Shape: (163, 1280)
Val Shape  : (41, 1280)

[Step 2] Training XGBoost

[Step 3] Evaluation

AUC       : 0.8405
F1 Score  : 0.8372
Accuracy  : 0.8293
Fold Time : 0.4 min

Fold 4/5

[Step 1] Feature Extraction


Train Shape: (163, 1280)
Val Shape  : (41, 1280)

[Step 2] Training XGBoost

[Step 3] Evaluation

AUC       : 0.9262
F1 Score  : 0.7778
Accuracy  : 0.8049
Fold Time : 0.4 min

Fold 5/5

[Step 1] Feature Extraction


...

  Epoch [07/10] Train Loss: 0.0257  Val Loss: 0.0243


  Epoch [08/10] Train Loss: 0.0242  Val Loss: 0.0164


  Epoch [09/10] Train Loss: 0.0297  Val Loss: 0.0228


  Epoch [10/10] Train Loss: 0.0230  Val Loss: 0.0228
  ✅ Best Val Loss: 0.0153

[Step 1B] Extracting Fusion Features


Train Shape: (164, 1664)
Val Shape  : (40, 1664)

[Step 2] Training XGBoost

[Step 3] Evaluation

AUC       : 1.0000
F1 Score  : 0.9474
Accuracy  : 0.9500
Fold Time : 4.3 min

------------------------------------------------------------
CONFIG COMPLETE: EfficientNet + ViT + Attention (proposed)
------------------------------------------------------------
Mean AUC  : 0.9962 ± 0.0065
Mean F1   : 0.9599 ± 0.0198
Avg Time  : 4.19 min/fold
Total CFG : 20.9 min

ABLATION TABLE (COPY INTO PAPER)
Configuration                                    Accuracy   Precision      Recall          F1     ROC_AUC         MCC       Kappa Specificity
---------------------------------------------------------------------------------------------------------------------------------------------
EfficientNetV2-S only                              0.7550      0.8117      0.6748      0.7310      0.8499      0.5234      0.5096      0.8352
ViT-S/16 only                                      0.9312      0.9794      0.882

                                                CELL 3

In [12]:
# =========================================================
# BASELINE MODEL COMPARISON
# ResNet50 vs DenseNet121 vs ViT-S/16
# 5-Fold Stratified Cross Validation
# Clean tqdm + Epoch Logging
# =========================================================

import os
import time
import timm
import torch
import random
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix
)

import torchvision.transforms as transforms

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================

DATASET_PATH = "/Users/shuvasishsaha/Documents/SAAMPLE"

SEED         = 42
NUM_FOLDS    = 5
BATCH_SIZE   = 4
EPOCHS       = 15
LR           = 1e-4
NUM_CLASSES  = 2

# =========================================================
# REPRODUCIBILITY
# =========================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================================================
# DEVICE
# =========================================================

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)

print(f"\nUsing Device: {device}")

# =========================================================
# TRANSFORMS
# =========================================================

train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomVerticalFlip(),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# =========================================================
# DATASET
# =========================================================

full_dataset = ImageFolder(DATASET_PATH)

targets      = full_dataset.targets
class_names  = full_dataset.classes

print(f"Classes      : {class_names}")
print(f"Total Images : {len(full_dataset)}")

# =========================================================
# MODELS
# =========================================================

BASELINES = [

    {
        "name": "ResNet50",
        "timm": "resnet50"
    },

    {
        "name": "DenseNet121",
        "timm": "densenet121"
    },

    {
        "name": "ViT-S/16",
        "timm": "vit_small_patch16_224"
    }
]

# =========================================================
# TRAIN FUNCTION
# =========================================================

def train_epoch(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0.0

    batch_bar = tqdm(
        loader,
        leave=False,
        ncols=110,
        bar_format='{l_bar}{bar:20}{r_bar}'
    )

    for images, labels in batch_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        batch_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(loader)

# =========================================================
# VALIDATION LOSS
# =========================================================

@torch.no_grad()
def validation_loss(model, loader, criterion):

    model.eval()

    total_loss = 0.0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        total_loss += loss.item()

    return total_loss / len(loader)

# =========================================================
# EVALUATION
# =========================================================

@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    all_preds  = []
    all_probs  = []
    all_labels = []

    for images, labels in loader:

        images = images.to(device)

        logits = model(images)

        probs = torch.softmax(logits, dim=1)[:, 1]

        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    return {

        "Accuracy":
            accuracy_score(y_true, y_pred),

        "Precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "ROC_AUC":
            roc_auc_score(y_true, y_prob),

        "MCC":
            matthews_corrcoef(y_true, y_pred),

        "Kappa":
            cohen_kappa_score(y_true, y_pred),

        "Specificity":
            tn / (tn + fp)
    }

# =========================================================
# CROSS VALIDATION
# =========================================================

kf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=SEED
)

all_rows   = []
param_info = {}

overall_start = time.time()

# =========================================================
# MODEL LOOP
# =========================================================

for baseline in BASELINES:

    name      = baseline["name"]
    timm_name = baseline["timm"]

    print("\n" + "="*70)
    print(f"BASELINE MODEL : {name}")
    print("="*70)

    model_start = time.time()

    fold_metrics = []
    fold_times   = []

    # =====================================================
    # FOLD LOOP
    # =====================================================

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(np.arange(len(targets)), targets)
    ):

        print(f"\nStarting Fold [{fold+1}/{NUM_FOLDS}]")

        fold_start = time.time()

        # =====================================================
        # MODEL
        # =====================================================

        model = timm.create_model(
            timm_name,
            pretrained=True,
            num_classes=NUM_CLASSES
        ).to(device)

        if fold == 0:

            n_params = (
                sum(p.numel() for p in model.parameters())
                / 1e6
            )

            param_info[name] = f"{n_params:.1f}M"

        # =====================================================
        # DATASET SPLITS
        # =====================================================

        train_sub = Subset(

            ImageFolder(
                DATASET_PATH,
                transform=train_transform
            ),

            train_idx
        )

        val_sub = Subset(

            ImageFolder(
                DATASET_PATH,
                transform=val_transform
            ),

            val_idx
        )

        # =====================================================
        # DATALOADERS
        # =====================================================

        t_loader = DataLoader(
            train_sub,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0
        )

        v_loader = DataLoader(
            val_sub,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        # =====================================================
        # LOSS / OPTIMIZER
        # =====================================================

        criterion = nn.CrossEntropyLoss()

        optimizer = optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=1e-4
        )

        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS
        )

        # =====================================================
        # TRAINING
        # =====================================================

        best_val_auc = -1.0
        best_state   = None

        for epoch in range(EPOCHS):

            # =================================================
            # TRAIN
            # =================================================

            train_loss = train_epoch(
                model,
                t_loader,
                optimizer,
                criterion
            )

            # =================================================
            # VALIDATION LOSS
            # =================================================

            val_loss = validation_loss(
                model,
                v_loader,
                criterion
            )

            # =================================================
            # METRICS
            # =================================================

            metrics = evaluate(model, v_loader)

            scheduler.step()

            current_auc = metrics["ROC_AUC"]

            # =================================================
            # CLEAN OUTPUT
            # =================================================

            print(
                f"Epoch [{epoch+1:02d}/{EPOCHS}] "
                f"Train Loss: {train_loss:.4f}  "
                f"Val Loss: {val_loss:.4f}"
            )

            # =================================================
            # SAVE BEST MODEL
            # =================================================

            if current_auc > best_val_auc:

                best_val_auc = current_auc

                best_state = {

                    k: v.clone()

                    for k, v in model.state_dict().items()
                }

        # =====================================================
        # LOAD BEST MODEL
        # =====================================================

        model.load_state_dict(best_state)

        final_metrics = evaluate(model, v_loader)

        fold_metrics.append(final_metrics)

        fold_time = time.time() - fold_start

        fold_times.append(fold_time)

        print(
            f"\nFold {fold+1} Completed | "
            f"AUC={final_metrics['ROC_AUC']:.4f} | "
            f"F1={final_metrics['F1']:.4f} | "
            f"Time={(fold_time/60):.2f} min"
        )

        # =====================================================
        # MEMORY CLEANUP
        # =====================================================

        del model

        if device.type == "mps":
            torch.mps.empty_cache()

    # =========================================================
    # AGGREGATE RESULTS
    # =========================================================

    metric_keys = list(fold_metrics[0].keys())

    avg = {

        k: np.mean([m[k] for m in fold_metrics])

        for k in metric_keys
    }

    std = {

        k: np.std([m[k] for m in fold_metrics])

        for k in metric_keys
    }

    row = {

        "Model": name,

        "Params": param_info[name]
    }

    for k in metric_keys:

        row[k] = round(avg[k], 4)

        row[f"{k}_std"] = round(std[k], 4)

    row["Avg_time_min"] = round(
        np.mean(fold_times) / 60,
        2
    )

    all_rows.append(row)

    model_total_time = (
        time.time() - model_start
    ) / 60

    # =========================================================
    # SUMMARY
    # =========================================================

    print("\n" + "-"*70)
    print(f"{name} SUMMARY")
    print("-"*70)

    for k in metric_keys:

        print(
            f"{k:<14}: "
            f"{avg[k]:.4f} ± {std[k]:.4f}"
        )

    print(f"\nTotal Training Time: {model_total_time:.2f} min")

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(all_rows)

results_df.to_csv(
    "baseline_results.csv",
    index=False
)

comparison_cols = [

    "Model",
    "Params",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "MCC",
    "Kappa",
    "Specificity",
    "Avg_time_min"
]

comparison_df = results_df[comparison_cols]

comparison_df.to_csv(
    "comparison_table.csv",
    index=False
)

# =========================================================
# FINAL OUTPUT
# =========================================================

overall_time = (
    time.time() - overall_start
) / 60

print("\n" + "="*80)
print("ALL BASELINE EXPERIMENTS COMPLETED")
print("="*80)

print(f"\nTotal Runtime: {overall_time:.2f} minutes")

print("\nSaved Files:")
print("1. baseline_results.csv")
print("2. comparison_table.csv")


Using Device: mps
Classes      : ['healthy', 'infected']
Total Images : 204

BASELINE MODEL : ResNet50

Starting Fold [1/5]


Epoch [01/15] Train Loss: 0.6907  Val Loss: 0.6919


Epoch [02/15] Train Loss: 0.6784  Val Loss: 0.6719


Epoch [03/15] Train Loss: 0.6573  Val Loss: 0.6495


Epoch [04/15] Train Loss: 0.6303  Val Loss: 0.6251


...

Epoch [11/15] Train Loss: 0.0081  Val Loss: 0.0054


Epoch [12/15] Train Loss: 0.0129  Val Loss: 0.0458


Epoch [13/15] Train Loss: 0.0032  Val Loss: 0.0184


Epoch [14/15] Train Loss: 0.0128  Val Loss: 0.0035


Epoch [15/15] Train Loss: 0.0042  Val Loss: 0.0029

Fold 5 Completed | AUC=1.0000 | F1=0.8571 | Time=7.81 min

----------------------------------------------------------------------
ViT-S/16 SUMMARY
----------------------------------------------------------------------
Accuracy      : 0.8872 ± 0.1461
Precision     : 0.9135 ± 0.1730
Recall        : 0.9300 ± 0.0980
F1            : 0.9057 ± 0.1047
ROC_AUC       : 1.0000 ± 0.0000
MCC           : 0.8036 ± 0.2475
Kappa         : 0.7712 ± 0.2980
Specificity   : 0.8400 ± 0.3200

Total Training Time: 52.09 min

ALL BASELINE EXPERIMENTS COMPLETED

Total Runtime: 214.74 minutes

Saved Files:
1. baseline_results.csv
2. comparison_table.csv


                                        CELL 4

In [ ]:
# =========================================================# ---------------------------------------------------------
# Uses:
#   - YOUR trained AttentionFusion layer
#   - Frozen EfficientNetV2-S backbone
#   - Proper Grad-CAM implementation
#   - Same architecture as training pipeline
#
# IMPORTANT:
# Since final prediction uses XGBoost (non-differentiable),
# Grad-CAM explains ONLY the CNN spatial attention regions,
# not the final XGBoost tree decisions.
#
# FIXED:
#   Correctly loads best_fusion_layer.pt
#   Recreates original AttentionFusion architecture
#   xtracts EfficientNet backbone properly
#   No state_dict mismatch error
# =========================================================

import os
import cv2
import timm
import torch
import random
import warnings
import numpy as np
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

from PIL import Image
from pathlib import Path

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================

SEED = 42

IMAGE_PATH = "/Users/shuvasishsaha/Documents/SAAMPLE/healthy/1000173521.jpg"

SAVE_PATH = "outputs/gradcam_final.png"

FUSION_MODEL_PATH = "outputs/models/best_fusion_layer.pt"

CLASS_NAMES = ["Healthy", "Disease"]

IMG_SIZE = 224

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================================================
# DEVICE
# =========================================================

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)

print(f"Device: {device}")

# =========================================================
# IMAGE TRANSFORM
# =========================================================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# =========================================================
# ATTENTION FUSION MODULE
# ---------------------------------------------------------
# EXACT SAME architecture used during training
# =========================================================

class AttentionFusion(nn.Module):

    def __init__(self, cnn_dim, vit_dim, num_classes=2):

        super().__init__()

        self.total_dim = cnn_dim + vit_dim

        self.attention = nn.Sequential(
            nn.Linear(self.total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, self.total_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.total_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, cnn_feat, vit_feat, return_logits=False):

        combined = torch.cat([cnn_feat, vit_feat], dim=1)

        weights = self.attention(combined)

        fused = combined * weights

        if return_logits:
            return self.classifier(fused)

        return fused

# =========================================================
# LOAD EFFICIENTNET BACKBONE
# =========================================================

print("\nLoading EfficientNetV2-S backbone...")

cnn_model = timm.create_model(
    "tf_efficientnetv2_s",
    pretrained=True,
    num_classes=0
)

cnn_model = cnn_model.to(device)
cnn_model.eval()

# =========================================================
# LOAD ViT BACKBONE
# =========================================================

vit_model = timm.create_model(
    "vit_small_patch16_224",
    pretrained=True,
    num_classes=0
)

vit_model = vit_model.to(device)
vit_model.eval()

# =========================================================
# FEATURE DIMENSIONS
# =========================================================

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():

    cnn_dim = cnn_model(dummy).shape[1]
    vit_dim = vit_model(dummy).shape[1]

print(f"\nCNN Feature Dim : {cnn_dim}")
print(f"ViT Feature Dim : {vit_dim}")

# =========================================================
# LOAD TRAINED FUSION LAYER
# =========================================================

fusion_layer = AttentionFusion(
    cnn_dim,
    vit_dim,
    num_classes=2
).to(device)

if os.path.exists(FUSION_MODEL_PATH):

    print("\nLoading trained fusion layer...")

    fusion_layer.load_state_dict(
        torch.load(
            FUSION_MODEL_PATH,
            map_location=device
        )
    )

    print("Fusion layer loaded successfully.")

else:

    print("\nWARNING:")
    print("Fusion model not found.")
    print("Using randomly initialized fusion layer.")

fusion_layer.eval()

# =========================================================
# SIMPLE CNN CLASSIFIER FOR GRADIENT FLOW
# ---------------------------------------------------------
# Needed because XGBoost is non-differentiable
#
# We use fusion classifier logits for gradients.
# =========================================================

class CNNGradCAMWrapper(nn.Module):

    def __init__(self, cnn_model, vit_model, fusion_layer):

        super().__init__()

        self.cnn_model = cnn_model
        self.vit_model = vit_model
        self.fusion_layer = fusion_layer

    def forward(self, x):

        cnn_feat = self.cnn_model(x)

        with torch.no_grad():
            vit_feat = self.vit_model(x)

        logits = self.fusion_layer(
            cnn_feat,
            vit_feat,
            return_logits=True
        )

        return logits

# =========================================================
# WRAPPED MODEL
# =========================================================

model = CNNGradCAMWrapper(
    cnn_model,
    vit_model,
    fusion_layer
).to(device)

model.eval()

# =========================================================
# TARGET LAYER
# ---------------------------------------------------------
# Better semantic localization layer
# =========================================================

target_layer = model.cnn_model.conv_head

# =========================================================
# GRAD-CAM CLASS
# =========================================================

class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        self.forward_hook = target_layer.register_forward_hook(
            self.save_activation
        )

        self.backward_hook = target_layer.register_full_backward_hook(
            self.save_gradient
        )

    # -----------------------------------------------------

    def save_activation(self, module, input, output):

        self.activations = output.detach()

    # -----------------------------------------------------

    def save_gradient(self, module, grad_input, grad_output):

        self.gradients = grad_output[0].detach()

    # -----------------------------------------------------

    def generate(self, input_tensor, class_idx=None):

        self.model.zero_grad()

        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        score = output[:, class_idx]

        score.backward()

        gradients = self.gradients

        activations = self.activations

        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        cam = (weights * activations).sum(
            dim=1,
            keepdim=True
        )

        cam = torch.relu(cam)

        cam = cam.squeeze()

        cam = cam.cpu().numpy()

        cam -= cam.min()

        cam /= (cam.max() + 1e-8)

        return cam, class_idx

# =========================================================
# IMAGE PREP
# =========================================================

pil_img = Image.open(IMAGE_PATH).convert("RGB")

original_np = np.array(
    pil_img.resize((IMG_SIZE, IMG_SIZE))
)

input_tensor = transform(pil_img).unsqueeze(0).to(device)

input_tensor.requires_grad_()

# =========================================================
# RUN GRAD-CAM
# =========================================================

gradcam = GradCAM(model, target_layer)

cam, pred_idx = gradcam.generate(input_tensor)

# =========================================================
# PREDICTION CONFIDENCE
# =========================================================

with torch.no_grad():

    logits = model(input_tensor)

    probs = torch.softmax(logits, dim=1)[0]

    confidence = probs[pred_idx].item()

pred_label = CLASS_NAMES[pred_idx]

print(f"\nPrediction : {pred_label}")
print(f"Confidence : {confidence:.4f}")

# =========================================================
# HEATMAP
# =========================================================

cam_resized = cv2.resize(
    cam,
    (IMG_SIZE, IMG_SIZE)
)

heatmap = cv2.applyColorMap(
    np.uint8(255 * cam_resized),
    cv2.COLORMAP_JET
)

heatmap = cv2.cvtColor(
    heatmap,
    cv2.COLOR_BGR2RGB
)

# =========================================================
# OVERLAY
# =========================================================

overlay = (
    0.5 * heatmap +
    0.5 * original_np
).astype(np.uint8)

# =========================================================
# VISUALIZATION
# =========================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

fig.suptitle(
    f"EfficientNetV2-S Grad-CAM\n"
    f"Prediction: {pred_label} | "
    f"Confidence: {confidence:.4f}",
    fontsize=14,
    fontweight="bold"
)

# ---------------------------------------------------------
# Original Image
# ---------------------------------------------------------

axes[0].imshow(original_np)
axes[0].set_title("Original Image")
axes[0].axis("off")

# ---------------------------------------------------------
# Heatmap
# ---------------------------------------------------------

axes[1].imshow(heatmap)
axes[1].set_title("Grad-CAM Heatmap")
axes[1].axis("off")

# ---------------------------------------------------------
# Overlay
# ---------------------------------------------------------

axes[2].imshow(overlay)
axes[2].set_title("Overlay Visualization")
axes[2].axis("off")

plt.tight_layout()

# =========================================================
# SAVE FIGURE
# =========================================================

Path(SAVE_PATH).parent.mkdir(
    parents=True,
    exist_ok=True
)

plt.savefig(
    SAVE_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(f"\nSaved Grad-CAM figure:")
print(SAVE_PATH)

# =========================================================
# CLEANUP
# =========================================================

gradcam.forward_hook.remove()
gradcam.backward_hook.remove()

print("\nGrad-CAM completed successfully.")

Device: mps

Loading EfficientNetV2-S backbone...

CNN Feature Dim : 1280
ViT Feature Dim : 384

Loading trained fusion layer...
Fusion layer loaded successfully.

Prediction : Healthy
Confidence : 0.9556

Saved Grad-CAM figure:
outputs/gradcam_final.png

Grad-CAM completed successfully.


                                            CELL 5

In [ ]:
# =========================================================# ---------------------------------------------------------
# Hybrid Explainability Framework
#
# Uses:
#   - EfficientNetV2-S backbone
#   - ViT-S/16 backbone
#   - Trained AttentionFusion layer
#   - Proper Grad-CAM implementation
#   - Publication-quality visualization
#
# IMPORTANT:
# Final prediction logits come from the differentiable
# fusion classifier, NOT XGBoost.
#
# Grad-CAM therefore explains:
#   - CNN spatial representations
#   - conditioned by fusion logits
#
# It does NOT explain:
#   - XGBoost tree decisions
#
# FIXED:
#   Correct fusion layer loading
#   No state_dict mismatch
#   Proper hybrid architecture
#   Proper gradient flow
#   Batch processing support
# =========================================================

import os
import cv2
import timm
import torch
import random
import warnings
import numpy as np
import torch.nn as nn
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import torchvision.transforms as transforms

from PIL import Image
from pathlib import Path

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================

SEED = 42

IMAGE_PATH = "/Users/shuvasishsaha/Documents/SAAMPLE/healthy/1000173521.jpg"

IMAGE_DIR = "/Users/shuvasishsaha/Documents/SAAMPLE"

SAVE_DIR = "outputs/gradcam_results"

FUSION_MODEL_PATH = "outputs/models/best_fusion_layer.pt"

CLASS_NAMES = ["Healthy", "Disease"]

IMG_SIZE = 224

BATCH_MAX_IMAGES = 20

# =========================================================
# REPRODUCIBILITY
# =========================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================================================
# DEVICE
# =========================================================

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)

print(f"\nUsing Device: {device}")

# =========================================================
# TRANSFORM
# =========================================================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# =========================================================
# LOAD EFFICIENTNETV2-S BACKBONE
# =========================================================

print("\nLoading EfficientNetV2-S backbone...")

cnn_model = timm.create_model(
    "tf_efficientnetv2_s",
    pretrained=True,
    num_classes=0
)

cnn_model = cnn_model.to(device)
cnn_model.eval()

# =========================================================
# LOAD VIT-S/16 BACKBONE
# =========================================================

print("\nLoading ViT-S/16 backbone...")

vit_model = timm.create_model(
    "vit_small_patch16_224",
    pretrained=True,
    num_classes=0
)

vit_model = vit_model.to(device)
vit_model.eval()

# =========================================================
# FEATURE DIMENSIONS
# =========================================================

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

with torch.no_grad():

    cnn_dim = cnn_model(dummy).shape[1]
    vit_dim = vit_model(dummy).shape[1]

print(f"\nCNN Feature Dim : {cnn_dim}")
print(f"ViT Feature Dim : {vit_dim}")

# =========================================================
# ATTENTION FUSION MODULE
# =========================================================

class AttentionFusion(nn.Module):

    def __init__(self, cnn_dim, vit_dim, num_classes=2):

        super().__init__()

        self.total_dim = cnn_dim + vit_dim

        self.attention = nn.Sequential(
            nn.Linear(self.total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, self.total_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.total_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, cnn_feat, vit_feat, return_logits=False):

        combined = torch.cat(
            [cnn_feat, vit_feat],
            dim=1
        )

        weights = self.attention(combined)

        fused = combined * weights

        if return_logits:

            return self.classifier(fused)

        return fused

# =========================================================
# LOAD TRAINED FUSION LAYER
# =========================================================

fusion_layer = AttentionFusion(
    cnn_dim,
    vit_dim,
    num_classes=2
).to(device)

if os.path.exists(FUSION_MODEL_PATH):

    print("\nLoading trained fusion layer...")

    fusion_layer.load_state_dict(
        torch.load(
            FUSION_MODEL_PATH,
            map_location=device
        )
    )

    print("Fusion layer loaded successfully.")

else:

    print("\nWARNING:")
    print("Fusion model not found.")
    print("Using randomly initialized fusion layer.")

fusion_layer.eval()

# =========================================================
# HYBRID MODEL WRAPPER
# =========================================================

class HybridGradCAMModel(nn.Module):

    def __init__(
        self,
        cnn_model,
        vit_model,
        fusion_layer
    ):

        super().__init__()

        self.cnn_model = cnn_model
        self.vit_model = vit_model
        self.fusion_layer = fusion_layer

    def forward(self, x):

        cnn_feat = self.cnn_model(x)

        with torch.no_grad():

            vit_feat = self.vit_model(x)

        logits = self.fusion_layer(
            cnn_feat,
            vit_feat,
            return_logits=True
        )

        return logits

# =========================================================
# FINAL HYBRID MODEL
# =========================================================

hybrid_model = HybridGradCAMModel(
    cnn_model,
    vit_model,
    fusion_layer
).to(device)

hybrid_model.eval()

# =========================================================
# TARGET LAYER
# ---------------------------------------------------------
# Final semantic CNN layer
# =========================================================

cnn_target_layer = hybrid_model.cnn_model.conv_head

# =========================================================
# GRAD-CAM
# =========================================================

class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        self.forward_hook = (
            target_layer.register_forward_hook(
                self.save_activation
            )
        )

        self.backward_hook = (
            target_layer.register_full_backward_hook(
                self.save_gradient
            )
        )

    # -----------------------------------------------------

    def save_activation(
        self,
        module,
        input,
        output
    ):

        self.activations = output.detach()

    # -----------------------------------------------------

    def save_gradient(
        self,
        module,
        grad_input,
        grad_output
    ):

        self.gradients = grad_output[0].detach()

    # -----------------------------------------------------

    def generate(
        self,
        input_tensor,
        class_idx=None
    ):

        self.model.zero_grad()

        output = self.model(input_tensor)

        if class_idx is None:

            class_idx = output.argmax(
                dim=1
            ).item()

        score = output[:, class_idx]

        score.backward()

        gradients = self.gradients
        activations = self.activations

        # =================================================
        # GLOBAL AVERAGE POOLING OF GRADIENTS
        # =================================================

        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        # =================================================
        # WEIGHTED ACTIVATION MAP
        # =================================================

        cam = (
            weights * activations
        ).sum(
            dim=1,
            keepdim=True
        )

        cam = torch.relu(cam)

        cam = cam.squeeze()

        cam = cam.cpu().numpy()

        # =================================================
        # NORMALIZATION
        # =================================================

        cam -= cam.min()

        cam /= (cam.max() + 1e-8)

        return cam, class_idx

    # -----------------------------------------------------

    def remove_hooks(self):

        self.forward_hook.remove()
        self.backward_hook.remove()

# =========================================================
# IMAGE PREPARATION
# =========================================================

def prepare_image(image_path):

    pil_img = Image.open(
        image_path
    ).convert("RGB")

    original_np = np.array(
        pil_img.resize((IMG_SIZE, IMG_SIZE))
    )

    input_tensor = transform(
        pil_img
    ).unsqueeze(0).to(device)

    input_tensor.requires_grad_()

    return (
        pil_img,
        original_np,
        input_tensor
    )

# =========================================================
# CREATE HEATMAP
# =========================================================

def create_heatmap(original_np, cam):

    cam_resized = cv2.resize(
        cam,
        (IMG_SIZE, IMG_SIZE)
    )

    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam_resized),
        cv2.COLORMAP_JET
    )

    heatmap = cv2.cvtColor(
        heatmap,
        cv2.COLOR_BGR2RGB
    )

    overlay = (
        0.5 * heatmap +
        0.5 * original_np
    ).astype(np.uint8)

    return heatmap, overlay

# =========================================================
# PREDICTION
# =========================================================

def get_prediction(model, input_tensor):

    with torch.no_grad():

        logits = model(input_tensor)

        probs = torch.softmax(
            logits,
            dim=1
        )[0]

        pred_idx = probs.argmax().item()

        confidence = probs[
            pred_idx
        ].item()

    return pred_idx, confidence

# =========================================================
# SAVE FIGURE
# =========================================================

def save_gradcam_figure(
    original_np,
    heatmap,
    overlay,
    pred_label,
    confidence,
    save_path,
    model_name
):

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14, 5)
    )

    fig.suptitle(
        f"{model_name} Grad-CAM\n"
        f"Prediction: {pred_label} | "
        f"Confidence: {confidence:.4f}",
        fontsize=14,
        fontweight="bold"
    )

    # =====================================================
    # ORIGINAL IMAGE
    # =====================================================

    axes[0].imshow(original_np)

    axes[0].set_title(
        "Original Image"
    )

    axes[0].axis("off")

    # =====================================================
    # HEATMAP
    # =====================================================

    axes[1].imshow(heatmap)

    axes[1].set_title(
        "Grad-CAM Heatmap"
    )

    axes[1].axis("off")

    # =====================================================
    # OVERLAY
    # =====================================================

    axes[2].imshow(overlay)

    axes[2].set_title(
        "Overlay Visualization"
    )

    axes[2].axis("off")

    plt.tight_layout()

    Path(save_path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

# =========================================================
# RUN SINGLE GRAD-CAM
# =========================================================

def run_gradcam(
    image_path,
    model,
    target_layer,
    model_name,
    save_path
):

    print(f"\nRunning Grad-CAM: {model_name}")

    (
        pil_img,
        original_np,
        input_tensor
    ) = prepare_image(image_path)

    gradcam = GradCAM(
        model,
        target_layer
    )

    cam, pred_idx = gradcam.generate(
        input_tensor
    )

    pred_idx, confidence = get_prediction(
        model,
        input_tensor
    )

    pred_label = CLASS_NAMES[pred_idx]

    heatmap, overlay = create_heatmap(
        original_np,
        cam
    )

    save_gradcam_figure(
        original_np,
        heatmap,
        overlay,
        pred_label,
        confidence,
        save_path,
        model_name
    )

    gradcam.remove_hooks()

    print(f"Saved: {save_path}")

    print(
        f"Prediction : {pred_label}"
    )

    print(
        f"Confidence : {confidence:.4f}"
    )

# =========================================================
# COMPARISON FIGURE
# =========================================================

def compare_models(
    image_path,
    save_path="outputs/gradcam_results/comparison.png"
):

    (
        pil_img,
        original_np,
        input_tensor
    ) = prepare_image(image_path)

    # =====================================================
    # HYBRID MODEL
    # =====================================================

    gradcam = GradCAM(
        hybrid_model,
        cnn_target_layer
    )

    cam, pred_idx = gradcam.generate(
        input_tensor
    )

    heatmap, overlay = create_heatmap(
        original_np,
        cam
    )

    pred_idx, confidence = get_prediction(
        hybrid_model,
        input_tensor
    )

    # =====================================================
    # FIGURE
    # =====================================================

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 5)
    )

    fig.suptitle(
        "Hybrid CNN-Transformer Explainability",
        fontsize=16,
        fontweight="bold"
    )

    axes[0].imshow(original_np)

    axes[0].set_title(
        "Original Image"
    )

    axes[0].axis("off")

    axes[1].imshow(heatmap)

    axes[1].set_title(
        "Grad-CAM Heatmap"
    )

    axes[1].axis("off")

    axes[2].imshow(overlay)

    axes[2].set_title(
        f"Overlay\n"
        f"Pred={CLASS_NAMES[pred_idx]} "
        f"({confidence:.3f})"
    )

    axes[2].axis("off")

    plt.tight_layout()

    Path(save_path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    gradcam.remove_hooks()

    print(
        f"\nComparison figure saved: "
        f"{save_path}"
    )

# =========================================================
# BATCH PROCESSING
# =========================================================

def batch_gradcam(
    image_dir,
    model,
    target_layer,
    model_name,
    out_dir,
    n_max=20
):

    image_dir = Path(image_dir)

    out_dir = Path(out_dir)

    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    exts = {
        '.jpg',
        '.jpeg',
        '.png',
        '.bmp'
    }

    images = [
        p for p in image_dir.rglob('*')
        if p.suffix.lower() in exts
    ][:n_max]

    print(
        f"\nRunning batch Grad-CAM "
        f"({model_name})"
    )

    print(f"Total Images: {len(images)}")

    for img_path in images:

        save_path = (
            out_dir /
            f"{img_path.stem}.png"
        )

        try:

            run_gradcam(
                image_path=str(img_path),
                model=model,
                target_layer=target_layer,
                model_name=model_name,
                save_path=str(save_path)
            )

        except Exception as e:

            print(
                f"Skipped {img_path.name}: {e}"
            )

# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    print("\n" + "="*60)

    print(
        "Q1-LEVEL HYBRID GRAD-CAM PIPELINE"
    )

    print("="*60)

    # =====================================================
    # SINGLE IMAGE
    # =====================================================

    run_gradcam(
        image_path=IMAGE_PATH,
        model=hybrid_model,
        target_layer=cnn_target_layer,
        model_name="Hybrid EfficientNetV2-S + ViT",
        save_path=f"{SAVE_DIR}/hybrid_gradcam.png"
    )

    # =====================================================
    # COMPARISON FIGURE
    # =====================================================

    compare_models(
        IMAGE_PATH,
        save_path=(
            f"{SAVE_DIR}/"
            f"hybrid_comparison.png"
        )
    )

    # =====================================================
    # BATCH PROCESSING
    # =====================================================

    batch_gradcam(
        image_dir=IMAGE_DIR,
        model=hybrid_model,
        target_layer=cnn_target_layer,
        model_name="Hybrid EfficientNetV2-S + ViT",
        out_dir=f"{SAVE_DIR}/batch_results",
        n_max=BATCH_MAX_IMAGES
    )

    print("\n" + "="*60)

    print(
        "GRAD-CAM ANALYSIS COMPLETED"
    )

    print("="*60)

    print("\nGenerated Outputs:")

    print("1. Hybrid Grad-CAM")
    print("2. Comparison Figure")
    print("3. Batch Qualitative Results")

    print("\nIMPORTANT NOTE:")

    print(
        "Grad-CAM explains the differentiable "
        "hybrid deep feature extractor prior "
        "to XGBoost classification."
    )


Using Device: mps

Loading EfficientNetV2-S backbone...

Loading ViT-S/16 backbone...

CNN Feature Dim : 1280
ViT Feature Dim : 384

Loading trained fusion layer...
Fusion layer loaded successfully.

Q1-LEVEL HYBRID GRAD-CAM PIPELINE

Running Grad-CAM: Hybrid EfficientNetV2-S + ViT
Saved: outputs/gradcam_results/hybrid_gradcam.png
Prediction : Healthy
Confidence : 0.9556

Comparison figure saved: outputs/gradcam_results/hybrid_comparison.png

Running batch Grad-CAM (Hybrid EfficientNetV2-S + ViT)
Total Images: 20

Running Grad-CAM: Hybrid EfficientNetV2-S + ViT
Saved: outputs/gradcam_results/batch_results/1000173552.png
Prediction : Healthy
Confidence : 0.9910

Running Grad-CAM: Hybrid EfficientNetV2-S + ViT
Saved: outputs/gradcam_results/batch_results/1000173546.png
Prediction : Healthy
Confidence : 0.9996

Running Grad-CAM: Hybrid EfficientNetV2-S + ViT
Saved: outputs/gradcam_results/batch_results/1000173547.png
Prediction : Healthy
Confidence : 0.9893

Running Grad-CAM: Hybrid Effi

                                            CELL 6

In [ ]:
# =========================================================
#
# Added Features:
# A. Deterministic inference
# B. Exception handling
# C. Batch inference support
# D. Save standardized prediction JSON (with Ground Truth)
# E. FLOPs / parameter reporting
#
# REQUIRES:
#   outputs/models/best_xgb_model.pkl
#   outputs/models/best_fusion_layer.pt
# =========================================================

import os
import json
import time
import random
import warnings

import torch
import timm
import joblib
import numpy as np
import torch.nn as nn

from PIL import Image
from pathlib import Path
from torchvision import transforms

warnings.filterwarnings("ignore")

# =========================================================
# A. DETERMINISTIC INFERENCE
# =========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# 1. DEVICE SETUP
# =========================================================

if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print(f"\nUsing Device: {device}")

# =========================================================
# 2. ATTENTION FUSION MODULE
# =========================================================

class AttentionFusion(nn.Module):

    def __init__(self, cnn_dim, vit_dim, num_classes=2):

        super().__init__()

        self.total_dim = cnn_dim + vit_dim

        self.attention = nn.Sequential(
            nn.Linear(self.total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, self.total_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.total_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, cnn_feat, vit_feat, return_logits=False):

        combined = torch.cat([cnn_feat, vit_feat], dim=1)

        weights = self.attention(combined)

        fused = combined * weights

        if return_logits:
            return self.classifier(fused)

        return fused

# =========================================================
# 3. LOAD BACKBONE MODELS
# =========================================================

print("\nLoading backbone models...")

cnn_model = timm.create_model(
    'tf_efficientnetv2_s',
    pretrained=True,
    num_classes=0
)

vit_model = timm.create_model(
    'vit_small_patch16_224',
    pretrained=True,
    num_classes=0
)

cnn_model.to(device).eval()
vit_model.to(device).eval()

print("Backbone models loaded.")

# =========================================================
# 4. FEATURE DIMENSIONS
# =========================================================

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():

    cnn_dim = cnn_model(dummy).shape[1]
    vit_dim = vit_model(dummy).shape[1]

print(f"\nCNN Feature Dim : {cnn_dim}")
print(f"ViT Feature Dim : {vit_dim}")

# =========================================================
# 5. LOAD TRAINED FUSION LAYER
# =========================================================

fusion_layer = AttentionFusion(cnn_dim, vit_dim).to(device)

fusion_layer.load_state_dict(
    torch.load(
        "outputs/models/best_fusion_layer.pt",
        map_location=device
    )
)

fusion_layer.eval()

print("Fusion layer loaded.")

# =========================================================
# 6. LOAD TRAINED XGBOOST MODEL
# =========================================================

xgb_model = joblib.load(
    "outputs/models/best_xgb_model.pkl"
)

print("XGBoost model loaded.")

# =========================================================
# 7. IMAGE TRANSFORM
# =========================================================

transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================================================
# 8. CLASS LABELS & EXPLICIT MAPPING
# =========================================================

classes = {
    0: "Healthy Leaf",
    1: "Disease Leaf"
}

# Used to clean and match ground truth folder naming conventions
folder_to_label_str = {
    "healthy": "Healthy Leaf",
    "infected": "Disease Leaf"
}

# =========================================================
# E. FLOPs + PARAMETER REPORTING
# =========================================================

try:
    from thop import profile

    flops_cnn, params_cnn = profile(
        cnn_model,
        inputs=(dummy,),
        verbose=False
    )

    flops_vit, params_vit = profile(
        vit_model,
        inputs=(dummy,),
        verbose=False
    )

    print("\n=================================")
    print("MODEL COMPLEXITY REPORT")
    print("=================================")

    print(f"EfficientNetV2-S FLOPs : {flops_cnn/1e9:.2f} GFLOPs")
    print(f"EfficientNetV2-S Params: {params_cnn/1e6:.2f} M")

    print(f"ViT-S/16 FLOPs         : {flops_vit/1e9:.2f} GFLOPs")
    print(f"ViT-S/16 Params        : {params_vit/1e6:.2f} M")

except ImportError:
    print("\nInstall THOP for FLOPs reporting:")
    print("pip install thop")

# =========================================================
# 9. LATENCY MEASUREMENT
# =========================================================

def measure_inference_latency(image_path, n_runs=50):

    times = []

    for i in range(n_runs + 10):

        t0 = time.time()

        _ = _predict(image_path)

        t1 = time.time()

        if i >= 10:
            times.append((t1 - t0) * 1000)

    mean_ms = np.mean(times)
    std_ms = np.std(times)

    print("\n=================================")
    print("INFERENCE LATENCY")
    print("=================================")

    print(f"{mean_ms:.2f} ± {std_ms:.2f} ms/image")

    return mean_ms, std_ms

# =========================================================
# 10. INTERNAL PREDICTION FUNCTION
# =========================================================

@torch.no_grad()
def _predict(image_path):
    """
    Returns:
        prediction, probability
    """
    try:
        image = Image.open(image_path).convert("RGB")

        tensor = transform(image).unsqueeze(0).to(device)

        cnn_feat = cnn_model(tensor)

        vit_feat = vit_model(tensor)

        fused = fusion_layer(
            cnn_feat,
            vit_feat,
            return_logits=False
        )

        feature_vector = fused.cpu().numpy()

        prediction = xgb_model.predict(feature_vector)[0]

        probability = xgb_model.predict_proba(feature_vector)[0]

        return prediction, probability

    except Exception as e:
        print(f"\nERROR during inference on {image_path}:")
        print(e)
        return None, None

# =========================================================
# 11. SINGLE IMAGE PREDICTION (STANDARDIZED JSON SCHEMA)
# =========================================================

def predict_image(image_path, save_json=True):

    prediction, probability = _predict(image_path)

    if prediction is None:
        return None

    confidence = np.max(probability)
    
    # Extract parent directory to derive structural ground truth
    parent_folder = Path(image_path).parent.name.lower()
    ground_truth = folder_to_label_str.get(parent_folder, "Unknown")

    # Q1-Ready Standardized JSON Schema
    result = {
        "image_path": str(image_path),
        "filename": Path(image_path).name,
        "ground_truth": ground_truth,
        "prediction": {
            "predicted_label": int(prediction),
            "predicted_class": classes[prediction],
            "confidence": float(confidence),
            "probabilities": {
                "healthy": float(probability[0]),
                "disease": float(probability[1])
            }
        },
        "execution_metadata": {
            "device_used": str(device),
            "timestamp_epoch": time.time()
        }
    }

    print("\n=================================")
    print("PREDICTION RESULT")
    print("=================================")
    print(f"Image           : {result['filename']}")
    print(f"Ground Truth    : {result['ground_truth']}")
    print(f"Predicted Class : {result['prediction']['predicted_class']}")
    print(f"Confidence      : {result['prediction']['confidence']:.4f}")

    # =====================================================
    # D. SAVE PREDICTION JSON
    # =====================================================
    if save_json:
        os.makedirs("outputs/predictions", exist_ok=True)

        json_path = (
            f"outputs/predictions/"
            f"{Path(image_path).stem}_prediction.json"
        )

        with open(json_path, "w") as f:
            json.dump(result, f, indent=4)

        print(f"\nPrediction JSON saved:\n{json_path}")

    return result

# =========================================================
# C. BATCH INFERENCE SUPPORT
# =========================================================

def predict_folder(folder_path):

    image_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

    image_paths = []

    for ext in image_extensions:
        image_paths.extend(Path(folder_path).glob(f"*{ext}"))
        # Support deep scanning if nested directories exist
        image_paths.extend(Path(folder_path).glob(f"**/*{ext}"))

    # Deduplicate paths found by glob patterns
    image_paths = sorted(list(set(image_paths)))

    print("\n=================================")
    print("BATCH INFERENCE")
    print("=================================")
    print(f"Total Images Discovered: {len(image_paths)}")

    all_results = []

    for idx, img_path in enumerate(image_paths):

        print(f"\n[{idx+1}/{len(image_paths)}] Processing: {img_path.name}")

        result = predict_image(str(img_path), save_json=False)

        if result is not None:
            all_results.append(result)

    # Save consolidated batch execution record
    os.makedirs("outputs/predictions", exist_ok=True)

    batch_json = "outputs/predictions/batch_predictions.json"

    with open(batch_json, "w") as f:
        json.dump(all_results, f, indent=4)

    print("\n=================================")
    print("BATCH INFERENCE COMPLETE")
    print("=================================")
    print(f"Consolidated batch metrics saved to: {batch_json}")

    return all_results

# =========================================================
# 12. RUN EXECUTION ENGINE
# =========================================================

if __name__ == "__main__":
    
    # Target validation single image
    target_image = "/Users/shuvasishsaha/Documents/SAAMPLE/healthy/1000173521.jpg"
    
    if os.path.exists(target_image):
        single_result = predict_image(target_image, save_json=True)
    else:
        print(f"\nSpecified baseline image path not found: {target_image}")

    # =========================================================
    # OPTIONAL BATCH EVALUATION PIPELINES (Uncomment as needed)
    # =========================================================
    # To run batch predictions over your complete directory workspace:
    # dataset_folder = "/Users/shuvasishsaha/Documents/SAAMPLE"
    # predict_folder(dataset_folder)

    # To benchmark runtime tracking profiles:
    # if os.path.exists(target_image):
    #     measure_inference_latency(target_image, n_runs=50)


Using Device: mps

Loading backbone models...
Backbone models loaded.

CNN Feature Dim : 1280
ViT Feature Dim : 384
Fusion layer loaded.
XGBoost model loaded.

Install THOP for FLOPs reporting:
pip install thop

PREDICTION RESULT
Image           : 1000173521.jpg
Ground Truth    : Healthy Leaf
Predicted Class : Healthy Leaf
Confidence      : 0.9681

Prediction JSON saved:
outputs/predictions/1000173521_prediction.json
